## Claim Extractor

In [5]:
"""
CASM — Claim Extractor (Hybrid Ensemble)
=========================================
Dual-pipeline:
  - Med7 (en_core_med7-trf)  → DRUG, DOSAGE, STRENGTH, FORM, FREQUENCY, ROUTE, DURATION
  - scispaCy (en_core_sci_md) → DISEASE, fallback conditions

Install:
    pip install spacy scispacy torch
    pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_md-0.5.4.tar.gz
    pip install https://huggingface.co/kormilitzin/en_core_med7_trf/resolve/main/en_core_med7_trf-3.4.2.1-py3-none-any.whl

NumPy compatibility note:
    spaCy/thinc require NumPy < 2.0. If you hit a binary incompatibility error:
        pip install "numpy==1.26" --reinstall
        pip install spacy --reinstall --no-cache
"""

import re
import spacy
import torch
from dataclasses import dataclass
from enum import Enum
from typing import Optional


# ─── Med7 clinical labels ─────────────────────────────────────────────────────

MED7_DRUG_LABELS    = {"DRUG"}
MED7_DOSAGE_LABELS  = {"DOSAGE", "STRENGTH"}
MED7_CLINICAL_NOISE = {"FREQUENCY", "DURATION", "FORM", "ROUTE"}
# ↑ These are REAL clinical info, but not "entities" for our Claim.entities list


# ─── Enums ────────────────────────────────────────────────────────────────────

class ClaimType(str, Enum):
    DOSAGE_CLAIM      = "DOSAGE_CLAIM"
    DRUG_SAFETY_CLAIM = "DRUG_SAFETY_CLAIM"
    DRUG_INTERACTION  = "DRUG_INTERACTION"
    PROCEDURAL_CLAIM  = "PROCEDURAL_CLAIM"
    DIAGNOSIS_CLAIM   = "DIAGNOSIS_CLAIM"
    POPULATION_CLAIM  = "POPULATION_CLAIM"
    GENERAL_MEDICAL   = "GENERAL_MEDICAL"


# ─── Data Classes ─────────────────────────────────────────────────────────────

@dataclass
class Entity:
    text:  str
    label: str   # DRUG | DOSAGE | STRENGTH | DISEASE | CONDITION | FORM | FREQUENCY | ROUTE | DURATION
    start: int
    end:   int


@dataclass
class Claim:
    claim_id:              str
    claim_text:            str
    claim_type:            ClaimType
    entities:              list[Entity]
    drug_names:            list[str]
    dosages:               list[str]
    conditions:            list[str]
    requires_verification: bool
    confidence:            float


# ─── Keyword Maps ─────────────────────────────────────────────────────────────

CLAIM_TYPE_KEYWORDS: dict[ClaimType, list[str]] = {
    ClaimType.DOSAGE_CLAIM: [
        "mg", "mcg", "ml", "%", "dose", "dosage", "twice", "once", "three times",
        "daily", "bid", "tid", "qid", "weekly", "monthly", "prescribe",
        "administer", "give", "start", "initiate", "tablet", "capsule", "drops",
    ],
    ClaimType.DRUG_SAFETY_CLAIM: [
        "avoid", "contraindicated", "do not use", "caution", "warning",
        "unsafe", "dangerous", "harmful", "risk", "adverse", "side effect",
        "toxicity", "nephrotoxic", "hepatotoxic", "cardiotoxic",
    ],
    ClaimType.DRUG_INTERACTION: [
        "interaction", "interacts", "combined with", "together with",
        "concurrent", "concomitant", "potentiates", "inhibits", "induces",
        "bleeding risk", "increased risk when",
    ],
    ClaimType.PROCEDURAL_CLAIM: [
        "monitor", "check", "test", "measure", "assess", "every", "weeks",
        "months", "follow up", "review", "screen", "scan", "blood test",
        "lab", "hba1c", "creatinine", "egfr", "blood pressure",
    ],
    ClaimType.DIAGNOSIS_CLAIM: [
        "diagnosis", "diagnose", "patient has", "presents with", "suffering from",
        "consistent with", "indicative of", "suggests", "confirms",
    ],
    ClaimType.POPULATION_CLAIM: [
        "elderly", "pediatric", "children", "pregnant", "renal", "kidney",
        "hepatic", "liver", "geriatric", "neonatal", "trimester",
    ],
}

SPLIT_CONJUNCTIONS = [
    r"\band also\b", r"\badditionally\b", r"\bfurthermore\b",
    r"\bin addition\b", r"\bmoreover\b", r"\bhowever\b", r"\balternatively\b",
]


# ─── Extractor ────────────────────────────────────────────────────────────────

class ClaimExtractor:
    """
    Hybrid Ensemble Claim Extractor.

    Pipeline 1 — Med7 (en_core_med7_trf):
        Handles DRUG, DOSAGE, STRENGTH, FORM, FREQUENCY, ROUTE, DURATION.
        High precision for clinical instructions.
        Transformer-based — benefits significantly from GPU acceleration.

    Pipeline 2 — scispaCy (en_core_sci_md):
        Fallback for DISEASE/CONDITION detection.
        Only disease-like entities are kept; generic ENTITY noise is discarded.

    GPU:
        If a CUDA-capable GPU is available, spaCy will automatically use it
        for the Med7 transformer pipeline. Call spacy.prefer_gpu() before
        loading any model (done automatically in __init__).
    """

    def __init__(
        self,
        med7_model:   str  = "en_core_med7_trf",
        sci_model:    str  = "en_core_sci_md",
        require_gpu:  bool = False,
    ):
        """
        Args:
            med7_model:  spaCy model name for Med7 (transformer).
            sci_model:   spaCy model name for scispaCy.
            require_gpu: If True, raises RuntimeError when no GPU is found.
                         Useful in production to prevent silent CPU fallback.
        """
        self._setup_device(require_gpu=require_gpu)

        print(f"[ClaimExtractor] Loading Med7 model    : {med7_model}")
        self.nlp_med7 = spacy.load(med7_model)

        print(f"[ClaimExtractor] Loading scispaCy model: {sci_model}")
        self.nlp_sci  = spacy.load(sci_model)

        print("[ClaimExtractor] Hybrid ensemble ready.")

    # ── Device Setup ──────────────────────────────────────────────────────────

    @staticmethod
    def _setup_device(require_gpu: bool = False) -> None:
        """
        Configure spaCy's compute device.

        Must be called BEFORE spacy.load() — spaCy allocates model tensors
        at load time and will not migrate them afterwards.

        Priority:
            1. CUDA GPU  (via PyTorch + spacy.prefer_gpu)
            2. CPU       (silent fallback unless require_gpu=True)

        Note on spaCy GPU=OFF:
            Med7 (en_core_med7_trf 3.4.x) was compiled against spaCy 3.4.
            Running it under spaCy 3.7 means the transformer backend may not
            activate GPU even when CUDA is available — this is a model/version
            mismatch, not a code error. The model still runs correctly on CPU.
            To get full GPU utilisation, retrain Med7 against spaCy 3.7 or
            wait for an official 3.7-compatible release.
        """
        if torch.cuda.is_available():
            activated = spacy.prefer_gpu()
            gpu_name  = torch.cuda.get_device_name(0)
            vram_gb   = torch.cuda.get_device_properties(0).total_memory / 1e9
            status    = "ON" if activated else "OFF (model version mismatch — see docstring)"
            print(
                f"[ClaimExtractor] GPU detected : {gpu_name} "
                f"({vram_gb:.1f} GB VRAM) — spaCy GPU={status}"
            )
        else:
            if require_gpu:
                raise RuntimeError(
                    "[ClaimExtractor] require_gpu=True but no CUDA GPU found. "
                    "Install CUDA + torch GPU build, or set require_gpu=False."
                )
            print("[ClaimExtractor] No GPU detected — running on CPU.")

    @staticmethod
    def device_info() -> dict:
        """Return a summary of the current compute environment."""
        return {
            "cuda_available":    torch.cuda.is_available(),
            "cuda_device_count": torch.cuda.device_count(),
            "cuda_device_name":  torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            "cuda_vram_gb":      round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
                                 if torch.cuda.is_available() else None,
            "spacy_gpu_active":  spacy.prefer_gpu() if torch.cuda.is_available() else False,
            "torch_version":     torch.__version__,
            "spacy_version":     spacy.__version__,
        }

    # ── Public API ────────────────────────────────────────────────────────────

    def extract(self, llm_response: str) -> list[Claim]:
        """Extract a list of Claim objects from raw LLM text."""
        sentences = self._split_into_sentences(llm_response)
        claims = []
        for idx, sentence in enumerate(sentences):
            if not sentence.strip():
                continue
            claim = self._process_sentence(sentence, idx)
            if claim:
                claims.append(claim)
        print(f"[ClaimExtractor] {len(claims)} claim(s) extracted.")
        return claims

    def extract_as_dict(self, llm_response: str) -> list[dict]:
        """Extract claims and return them as a list of plain dicts."""
        return [self._to_dict(c) for c in self.extract(llm_response)]

    # ── Sentence Splitting ────────────────────────────────────────────────────

    def _split_into_sentences(self, text: str) -> list[str]:
        # Use scispaCy for sentence boundary detection
        doc = self.nlp_sci(text)
        sentences = [sent.text.strip() for sent in doc.sents]
        final = []
        for sentence in sentences:
            final.extend(self._split_on_conjunctions(sentence))
        return [s for s in final if len(s.split()) >= 4]

    def _split_on_conjunctions(self, sentence: str) -> list[str]:
        parts = [sentence]
        for pattern in SPLIT_CONJUNCTIONS:
            new_parts = []
            for part in parts:
                split = re.split(pattern, part, flags=re.IGNORECASE)
                new_parts.extend([s.strip() for s in split if s.strip()])
            parts = new_parts
        return parts

    # ── Claim Processing ──────────────────────────────────────────────────────

    def _process_sentence(self, sentence: str, idx: int) -> Optional[Claim]:
        # ── Pipeline 1: Med7 — drugs, dosages, clinical structure ─────────────
        doc_med7 = self.nlp_med7(sentence)
        med7_entities = [
            Entity(
                text=ent.text,
                label=ent.label_,
                start=ent.start_char,
                end=ent.end_char,
            )
            for ent in doc_med7.ents
        ]

        # ── Pipeline 2: scispaCy — diseases & conditions only ─────────────────
        doc_sci = self.nlp_sci(sentence)
        DISEASE_LABELS = {"DISEASE", "CONDITION", "DISORDER"}
        sci_disease_entities = [
            Entity(
                text=ent.text,
                label=ent.label_,
                start=ent.start_char,
                end=ent.end_char,
            )
            for ent in doc_sci.ents
            if ent.label_ in DISEASE_LABELS
        ]

        # ── Merge: Med7 authoritative; scispaCy fills disease gaps ────────────
        med7_spans    = {(e.start, e.end) for e in med7_entities}
        merged_entities = list(med7_entities)
        for e in sci_disease_entities:
            if (e.start, e.end) not in med7_spans:
                merged_entities.append(e)

        drug_names = self._extract_drugs(med7_entities, sentence)
        dosages    = self._extract_dosages(med7_entities, sentence)
        conditions = self._extract_conditions(sci_disease_entities, sentence)
        claim_type = self._classify_claim_type(sentence, merged_entities)

        if claim_type == ClaimType.GENERAL_MEDICAL and not merged_entities:
            return None

        requires_verification = claim_type in {
            ClaimType.DOSAGE_CLAIM,
            ClaimType.DRUG_SAFETY_CLAIM,
            ClaimType.DRUG_INTERACTION,
            ClaimType.POPULATION_CLAIM,
        }

        return Claim(
            claim_id=f"claim_{idx}",
            claim_text=sentence,
            claim_type=claim_type,
            entities=merged_entities,
            drug_names=drug_names,
            dosages=dosages,
            conditions=conditions,
            requires_verification=requires_verification,
            confidence=0.0,
        )

    # ── Claim Type Classification ─────────────────────────────────────────────

    def _classify_claim_type(
        self, sentence: str, entities: list[Entity]
    ) -> ClaimType:
        sentence_lower = sentence.lower()
        scores: dict[ClaimType, int] = {ct: 0 for ct in ClaimType}

        for claim_type, keywords in CLAIM_TYPE_KEYWORDS.items():
            for keyword in keywords:
                if keyword in sentence_lower:
                    scores[claim_type] += 1

        best = max(scores, key=lambda ct: scores[ct])
        return best if scores[best] > 0 else ClaimType.GENERAL_MEDICAL

    # ── Entity Extraction ─────────────────────────────────────────────────────

    @staticmethod
    def _clean_drug_name(text: str) -> str:
        """
        Strip artefacts Med7 sometimes includes in drug spans:
          - Parenthetical abbreviations:  'Fluorometholone (FML)' → 'Fluorometholone'
          - Trailing/leading punctuation, brackets, whitespace
        """
        # Remove parenthetical suffix:  'Drug (ABBR)' → 'Drug'
        text = re.sub(r'\s*\(.*?\)\s*$', '', text)
        # Strip remaining leading/trailing non-alphanumeric characters
        text = re.sub(r'^[^A-Za-z]+|[^A-Za-z0-9]+$', '', text)
        return text.strip()

    def _extract_drugs(
        self, med7_entities: list[Entity], sentence: str
    ) -> list[str]:
        # Primary: Med7 DRUG label — high precision
        drugs = [
            self._clean_drug_name(e.text)
            for e in med7_entities
            if e.label in MED7_DRUG_LABELS
        ]
        drugs = [d for d in drugs if d]  # drop empty strings after cleaning

        # Secondary: regex suffix fallback (catches brand names Med7 misses)
        regex = re.findall(
            r'\b[A-Z][a-z]+(?:in|ol|ine|ide|ate|mab|nib|pril|sartan|statin|mycin|cillin|zole|olone)\b',
            sentence,
        )
        for d in regex:
            if d not in drugs:
                drugs.append(d)

        return list(dict.fromkeys(drugs))  # deduplicate, preserve order

    def _extract_dosages(
        self, med7_entities: list[Entity], sentence: str
    ) -> list[str]:
        # Primary: Med7 DOSAGE / STRENGTH labels
        dosages = [e.text for e in med7_entities if e.label in MED7_DOSAGE_LABELS]

        # Secondary: regex — catches formats Med7 might miss
        regex = re.findall(
            r'\b\d+(?:\.\d+)?\s*(?:mg|mcg|ml|g|%|units?|IU|mmol)(?:/(?:day|kg|dose))?\b',
            sentence,
            re.IGNORECASE,
        )
        for d in regex:
            if d.strip() not in dosages:
                dosages.append(d.strip())

        return list(dict.fromkeys(dosages))

    def _extract_conditions(
        self, sci_entities: list[Entity], sentence: str
    ) -> list[str]:
        # Primary: scispaCy disease labels only (no ENTITY noise)
        conditions = [e.text for e in sci_entities]

        # Secondary: known clinical abbreviations
        abbrevs = re.findall(
            r'\b(?:CKD|DM|T2DM|HTN|CAD|CHF|COPD|AF|DVT|PE|UTI|PRK|TransPRK)\b',
            sentence,
        )
        for a in abbrevs:
            if a not in conditions:
                conditions.append(a)

        return list(dict.fromkeys(conditions))

    # ── Serialisation ─────────────────────────────────────────────────────────

    def _to_dict(self, claim: Claim) -> dict:
        return {
            "claim_id":              claim.claim_id,
            "claim_text":            claim.claim_text,
            "claim_type":            claim.claim_type.value,
            "entities":              [
                {"text": e.text, "label": e.label} for e in claim.entities
            ],
            "drug_names":            claim.drug_names,
            "dosages":               claim.dosages,
            "conditions":            claim.conditions,
            "requires_verification": claim.requires_verification,
            "confidence":            claim.confidence,
        }


## Claim Extractor Test

In [6]:
testdict = ClaimExtractor()
zb = testdict.extract_as_dict(llm_response="To treat corneal haze 5 months post-TransPRK, prescribe Fluorometholone (FML) 0.1% eye drops four times daily for 4 weeks. You may also consider Oral Vitamin C (1000mg/day) to support corneal healing.")

[ClaimExtractor] GPU detected : NVIDIA GeForce GTX 1650 (4.3 GB VRAM) — spaCy GPU=OFF (model version mismatch — see docstring)
[ClaimExtractor] Loading Med7 model    : en_core_med7_trf
[ClaimExtractor] Loading scispaCy model: en_core_sci_md
[ClaimExtractor] Hybrid ensemble ready.
[ClaimExtractor] 2 claim(s) extracted.


In [7]:
zb

[{'claim_id': 'claim_0',
  'claim_text': 'To treat corneal haze 5 months post-TransPRK, prescribe Fluorometholone (FML) 0.1% eye drops four times daily for 4 weeks.',
  'claim_type': 'DOSAGE_CLAIM',
  'entities': [{'text': 'Fluorometholone (', 'label': 'DRUG'},
   {'text': '0.1%', 'label': 'STRENGTH'},
   {'text': 'eye', 'label': 'ROUTE'},
   {'text': 'drops', 'label': 'FORM'},
   {'text': 'four times daily', 'label': 'FREQUENCY'},
   {'text': 'for 4 weeks', 'label': 'DURATION'}],
  'drug_names': ['Fluorometholone'],
  'dosages': ['0.1%'],
  'conditions': ['TransPRK'],
  'requires_verification': True,
  'confidence': 0.0},
 {'claim_id': 'claim_1',
  'claim_text': 'You may also consider Oral Vitamin C (1000mg/day) to support corneal healing.',
  'claim_type': 'DOSAGE_CLAIM',
  'entities': [{'text': 'Oral', 'label': 'ROUTE'},
   {'text': 'Vitamin C', 'label': 'DRUG'},
   {'text': '1000mg/day', 'label': 'STRENGTH'}],
  'drug_names': ['Vitamin C', 'Vitamin'],
  'dosages': ['1000mg/day'],
 

In [8]:
test2 = testdict.extract_as_dict(llm_response="For a patient with a confirmed UTI, prescribe Trimethoprim 200mg twice daily for 7 days. If symptoms persist after 48 hours, switch to Nitrofurantoin 100mg four times daily for 5 days. Monitor renal function with creatinine levels weekly.")

C:\Users\basim\PycharmProjects\PythonProject\.venv\Lib\site-packages\thinc\shims\pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):


[ClaimExtractor] 3 claim(s) extracted.


In [9]:
test2

[{'claim_id': 'claim_0',
  'claim_text': 'For a patient with a confirmed UTI, prescribe Trimethoprim 200mg twice daily for 7 days.',
  'claim_type': 'DOSAGE_CLAIM',
  'entities': [{'text': 'Trimethoprim', 'label': 'DRUG'},
   {'text': '200mg', 'label': 'STRENGTH'},
   {'text': 'twice daily', 'label': 'FREQUENCY'},
   {'text': 'for 7 days', 'label': 'DURATION'}],
  'drug_names': ['Trimethoprim'],
  'dosages': ['200mg'],
  'conditions': ['UTI'],
  'requires_verification': True,
  'confidence': 0.0},
 {'claim_id': 'claim_1',
  'claim_text': 'If symptoms persist after 48 hours, switch to Nitrofurantoin 100mg four times daily for 5 days.',
  'claim_type': 'DOSAGE_CLAIM',
  'entities': [{'text': 'Nitrofurantoin', 'label': 'DRUG'},
   {'text': '100mg', 'label': 'STRENGTH'},
   {'text': 'four times daily', 'label': 'FREQUENCY'},
   {'text': 'for 5 days', 'label': 'DURATION'}],
  'drug_names': ['Nitrofurantoin'],
  'dosages': ['100mg'],
  'conditions': [],
  'requires_verification': True,
  'co

In [10]:
test3 = testdict.extract_as_dict(llm_response="In a T2DM patient with CKD stage 3, avoid Metformin 500mg if eGFR drops below 30 ml/min due to risk of lactic acidosis. Consider switching to Sitagliptin 50mg once daily, which is renally dosed and safer in this population. Monitor HbA1c every 3 months.")

[ClaimExtractor] 3 claim(s) extracted.


In [11]:
test3

[{'claim_id': 'claim_0',
  'claim_text': 'In a T2DM patient with CKD stage 3, avoid Metformin 500mg if eGFR drops below 30 ml/min due to risk of lactic acidosis.',
  'claim_type': 'DOSAGE_CLAIM',
  'entities': [{'text': 'Metformin', 'label': 'DRUG'},
   {'text': '500mg', 'label': 'STRENGTH'}],
  'drug_names': ['Metformin'],
  'dosages': ['500mg', '30 ml'],
  'conditions': ['T2DM', 'CKD'],
  'requires_verification': True,
  'confidence': 0.0},
 {'claim_id': 'claim_1',
  'claim_text': 'Consider switching to Sitagliptin 50mg once daily, which is renally dosed and safer in this population.',
  'claim_type': 'DOSAGE_CLAIM',
  'entities': [{'text': 'Sitagliptin', 'label': 'DRUG'},
   {'text': '50mg', 'label': 'STRENGTH'},
   {'text': 'once daily', 'label': 'FREQUENCY'}],
  'drug_names': ['Sitagliptin'],
  'dosages': ['50mg'],
  'conditions': [],
  'requires_verification': True,
  'confidence': 0.0},
 {'claim_id': 'claim_2',
  'claim_text': 'Monitor HbA1c every 3 months.',
  'claim_type': 'PR